# Практическое занятие 4. Прогноз температуры электропривода высотной платформы

Цель занятия - построить и сравнить простую физически
интерпретируемую модель, линейную регуляризованную модель,
случайный лес и градиентный бустинг для прогноза температуры
обмотки электродвигателя.

HAPS (High Altitude Platform Station) - высотная
псевдоспутниковая платформа. В инженерном смысле это летательный
аппарат длительного пребывания в стратосфере. Для такого объекта
тепловой режим электропривода важен из-за ограниченных запасов
массы, энергии и охлаждения.

**Задача студента.** Не требуется писать сложный код с нуля.
Необходимо последовательно выполнить ячейки, заполнить несколько
явно отмеченных учебных параметров, сравнить модели и объяснить,
какие признаки связаны с нагревом, а какие столбцы являются
диагностическими и не должны попадать в базовую модель.

## Инициализация среды выполнения

Ячейка ниже обеспечивает запуск блокнота в Google Colab и в
локальном Jupyter Notebook. Если проект уже открыт локально,
повторное клонирование не выполняется.

In [ ]:
# COLAB_BOOTSTRAP_APPailab
from pathlib import Path
import os
import sys

REQUIRED_PROCESSED_FILES = ['practice_04_haps_thermal_features.csv', 'practice_04_haps_thermal_diagnostics.csv', 'practice_04_06_dataset_catalog.csv', 'practice_04_06_dataset_assignments.csv']
PROJECT_REPOSITORY_URL = "https://github.com/Alexflex/appailab.git"

def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-colab.txt").exists() and (candidate / "data" / "processed").exists():
            return candidate
    return None

project_root = find_project_root(Path.cwd())

if project_root is None:
    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

    if IN_COLAB:
        workdir = Path("/content/appailab")
        if not workdir.exists():
            !git clone -q {PROJECT_REPOSITORY_URL} {workdir}
        project_root = workdir
        os.chdir(project_root)
        !pip install -q -r requirements-colab.txt
    else:
        raise FileNotFoundError(
            "Не найден корень проекта. Откройте блокнот из репозитория appailab "
            "или выполните git clone перед запуском."
        )

sys.path.insert(0, str(project_root / "src"))
missing_files = [
    name for name in REQUIRED_PROCESSED_FILES
    if not (project_root / "data" / "processed" / name).exists()
]
if missing_files:
    raise FileNotFoundError(
        "Не найдены подготовленные CSV: " + ", ".join(missing_files)
        + ". Выполните python scripts/generate_datasets.py."
    )

print(f"Корень проекта: {project_root}")
print("Проверенные CSV:", ", ".join(REQUIRED_PROCESSED_FILES))

In [ ]:
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

DATA_DIR = project_root / "data" / "processed"
CATALOG_04_06_FILE = DATA_DIR / "practice_04_06_dataset_catalog.csv"
ASSIGNMENTS_04_06_FILE = DATA_DIR / "practice_04_06_dataset_assignments.csv"
RANDOM_STATE = 20260507

## Источники и проверка актуальности

Для занятий 4-6 используются базовые локальные учебные CSV, поэтому
выполнение блокнота не зависит от загрузки внешних архивов. Открытые
источники ниже используются как научно-методические ориентиры для
расширенных заданий и проверки переносимости постановки.

1. NASA C-MAPSS Aircraft Engine Simulator Data - открытый набор
   траекторий деградации авиационных двигателей. Применение:
   временная регрессия, риск утечки между соседними точками,
   кластеризация режимов деградации. URL:
   https://data.nasa.gov/dataset/groups/c-mapss-aircraft-engine-simulator-data
2. Mendeley Data `Partial Discharge Signals in Insulated Power
   Cables with Time-of-Arrival Annotations` - временные сигналы
   частичных разрядов с аннотациями времени прихода импульсов.
   Применение: извлечение PRPD-признаков перед классификацией. URL:
   https://data.mendeley.com/datasets/3mdgxv6zt7
3. UCI `AI4I 2020 Predictive Maintenance Dataset` - промышленно
   мотивированный набор для предиктивного обслуживания. Применение:
   отделение сенсорных признаков от служебных кодов и признаков
   отказов перед кластеризацией. URL:
   https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset
4. NASA Prognostics Center of Excellence Data Repository - реестр
   наборов для диагностики и прогнозирования технического состояния.
   Применение: расширение задач кластеризации и анализа временных
   сценариев. URL:
   https://www.nasa.gov/content/prognostics-center-of-excellence-data-set-repository

Проверка ссылок выполнена 2026-05-15. Перед использованием полного
внешнего источника в самостоятельной работе необходимо повторно
проверить карточку набора данных, лицензию, размер архива и формат
файлов.

## Теоретический блок

Тепловая модель первого порядка описывает инерционное изменение
температуры:

$$
T_{k+1} = T_k + \frac{\Delta t}{\tau}
\left(T_{amb,k} + R_{th} P_{loss,k} - T_k\right),
$$

где `T` - температура обмотки, `T_amb` - температура окружающей
среды, `R_th` - тепловое сопротивление, `tau` - тепловая
постоянная времени, `P_loss` - суммарные потери.

Случайный лес (Random Forest) - ансамбль деревьев решений,
усредняющий прогнозы многих деревьев. Градиентный бустинг
(Gradient Boosting) - последовательный ансамбль, в котором каждая
следующая модель уточняет ошибки предыдущих. Оба метода полезны
для нелинейных зависимостей, но требуют контроля переобучения и
осторожной интерпретации важности признаков.

## Последовательность работы

В этой работе используется воспроизводимый pipeline
(последовательность обработки данных):

1. загрузить feature-CSV и diagnostics-CSV;
2. проверить физический смысл столбцов;
3. построить разведочные графики;
4. выбрать безопасные признаки;
5. обучить физическую базовую модель и модели машинного обучения;
6. сравнить метрики и остатки;
7. разобрать антипример утечки данных;
8. сформулировать инженерный вывод.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.axis("off")
boxes = [
    ("Электрическая\nмощность UI", 0.05),
    ("Полезная\nмеханическая\nмощность Mω", 0.27),
    ("Потери\nP_loss", 0.49),
    ("Нагрев\nобмотки", 0.68),
    ("Охлаждение\nвоздушным\nпотоком", 0.86),
]
for text, x in boxes:
    ax.text(
        x,
        0.55,
        text,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.45", facecolor="#f4f7fb", edgecolor="#4c78a8"),
        transform=ax.transAxes,
    )
for x0, x1 in [(0.14, 0.22), (0.36, 0.44), (0.58, 0.64), (0.77, 0.82)]:
    ax.annotate(
        "",
        xy=(x1, 0.55),
        xytext=(x0, 0.55),
        xycoords=ax.transAxes,
        arrowprops=dict(arrowstyle="->", lw=1.8, color="#333333"),
    )
ax.text(
    0.49,
    0.18,
    "Учебный прокси-признак: P_loss_proxy = U I - Mω",
    ha="center",
    va="center",
    transform=ax.transAxes,
    fontsize=12,
)
ax.set_title("Схема теплового баланса электропривода")
plt.show()

In [ ]:
FEATURES_FILE = DATA_DIR / "practice_04_haps_thermal_features.csv"
DIAGNOSTICS_FILE = DATA_DIR / "practice_04_haps_thermal_diagnostics.csv"

df = pd.read_csv(FEATURES_FILE)
diagnostics_df = pd.read_csv(DIAGNOSTICS_FILE)
full_df = df.merge(diagnostics_df, on="sample_id", validate="one_to_one")

display(df.head())
print("Размер feature-таблицы:", df.shape)
print("Размер diagnostics-таблицы:", diagnostics_df.shape)

## Паспорт набора данных

`feature-CSV` содержит только признаки, допустимые для базового
моделирования, и целевую переменную `winding_temp_c`.
`diagnostics-CSV` содержит расчетные тепловые величины:
суммарные потери, тепловое сопротивление, постоянную времени,
стационарную температуру и запас до теплового предела.

В базовую модель запрещено автоматически включать
`steady_state_temp_c`, `temperature_margin_c` и `is_overheated`,
так как эти столбцы прямо раскрывают способ формирования тепловой
цели или ее ограничений.

Важное правило занятия: diagnostics-CSV разрешается использовать
для объяснения физики и проверки выводов, но не как автоматический
источник входных признаков. Такое разделение защищает модель от
утечки данных (data leakage), то есть попадания в признаки
информации, которая недоступна в реальной задаче прогноза.

In [ ]:
display(df.describe().T)
print("Пропуски по столбцам:")
display(df.isna().sum().to_frame("missing_count"))
print("Диапазон температуры обмотки:")
display(df["winding_temp_c"].describe())

## Разведочный анализ данных

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

axes[0].hist(df["winding_temp_c"], bins=28, color="#4c78a8", edgecolor="white")
axes[0].set_title("Распределение температуры обмотки")
axes[0].set_xlabel("Температура, deg_C")

axes[1].scatter(df["current_a"], df["winding_temp_c"], alpha=0.65, s=18)
axes[1].set_title("Температура и ток")
axes[1].set_xlabel("Ток, A")
axes[1].set_ylabel("Температура, deg_C")

axes[2].scatter(df["cooling_air_speed_mps"], df["winding_temp_c"], alpha=0.65, s=18)
axes[2].set_title("Температура и скорость охлаждающего потока")
axes[2].set_xlabel("Скорость потока, m/s")
axes[2].set_ylabel("Температура, deg_C")

profile_means = df.groupby("profile_id")["winding_temp_c"].mean()
axes[3].bar(profile_means.index.astype(str), profile_means.values, color="#72b7b2")
axes[3].set_title("Средняя температура по профилям")
axes[3].set_xlabel("profile_id")
axes[3].set_ylabel("Температура, deg_C")

plt.tight_layout()
plt.show()

На следующем графике показана динамика температуры по профилям.
Это не полноценный временной ряд полета, но упорядоченность
`time_s` внутри каждого `profile_id` позволяет увидеть тепловую
инерцию: температура меняется плавнее, чем электрическая нагрузка.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for profile_id, group in df.groupby("profile_id"):
    ax.plot(group["time_s"], group["winding_temp_c"], label=f"profile {profile_id}", linewidth=1.8)
ax.set_title("Профили температуры обмотки во времени")
ax.set_xlabel("Время внутри профиля, s")
ax.set_ylabel("Температура обмотки, deg_C")
ax.legend(ncol=3)
plt.tight_layout()
plt.show()

График ниже использует `total_loss_w` из diagnostics-CSV только
для интерпретации. В реальном прогнозе эта величина может быть
неизвестна заранее или рассчитана из скрытых параметров объекта.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    full_df["total_loss_w"],
    full_df["winding_temp_c"],
    c=full_df["cooling_air_speed_mps"],
    cmap="viridis",
    s=24,
    alpha=0.75,
)
ax.set_title("Температура, потери и охлаждающий поток")
ax.set_xlabel("Суммарные потери из diagnostics, W")
ax.set_ylabel("Температура обмотки, deg_C")
fig.colorbar(scatter, ax=ax, label="Скорость охлаждающего потока, m/s")
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax, label="Коэффициент корреляции")
ax.set_title("Корреляционная матрица признаков занятия 4")
plt.tight_layout()
plt.show()

## Выбор признаков и разбиение выборки

Для начального занятия используется случайное разбиение с
сохранением доли каждого `profile_id` в обучающей и тестовой
выборках. Это упрощает сравнение моделей. В задачах строгого
прогнозирования временных рядов следует дополнительно проверять
групповое или хронологическое разбиение, чтобы соседние точки
одного испытания не создавали утечку информации.

In [ ]:
# TODO: заполните список признаков. используйте только измеряемые признаки из feature-CSV
# Рекомендуемые признаки: ['altitude_m', 'air_density_kg_m3', 'ambient_temp_c', 'cooling_air_speed_mps', 'speed_rpm', 'torque_nm', 'voltage_v', 'current_a']
thermal_features = None
if thermal_features is None:
    raise ValueError('Заполните thermal_features: используйте только измеряемые признаки из feature-CSV')

target_column = "winding_temp_c"
forbidden_columns = {"winding_temp_c", "steady_state_temp_c", "temperature_margin_c", "is_overheated"}
leaked = forbidden_columns.intersection(thermal_features)
if leaked:
    raise ValueError(f"Обнаружена утечка данных: {sorted(leaked)}")

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df["profile_id"],
)

X_train = df.loc[train_idx, thermal_features]
X_test = df.loc[test_idx, thermal_features]
y_train = df.loc[train_idx, target_column]
y_test = df.loc[test_idx, target_column]

print("Train:", X_train.shape, "Test:", X_test.shape)
display(df.loc[test_idx, "profile_id"].value_counts().sort_index().to_frame("test_count"))

## Проверка риска утечки между близкими режимами

Случайное разбиение удобно для первого сравнения моделей, но оно
может завышать качество, если соседние точки одного профиля
одновременно попадают в обучение и тест. Поэтому ниже
дополнительно показано групповое разбиение: часть `profile_id`
полностью оставляется для теста.

In [ ]:
group_test_profiles = [5, 6]
group_train_mask = ~df["profile_id"].isin(group_test_profiles)
group_test_mask = df["profile_id"].isin(group_test_profiles)

split_diagnostics = []
for split_name, train_index, test_index in [
    ("random_stratified", train_idx, test_idx),
    ("group_holdout", df.index[group_train_mask], df.index[group_test_mask]),
]:
    split_model = RandomForestRegressor(
        n_estimators=160,
        max_depth=8,
        min_samples_leaf=4,
        random_state=RANDOM_STATE,
    )
    split_model.fit(df.loc[train_index, thermal_features], df.loc[train_index, target_column])
    split_pred = split_model.predict(df.loc[test_index, thermal_features])
    split_diagnostics.append(
        {
            "split": split_name,
            "test_size": len(test_index),
            "RMSE_deg_C": np.sqrt(mean_squared_error(df.loc[test_index, target_column], split_pred)),
            "R2": r2_score(df.loc[test_index, target_column], split_pred),
        }
    )
split_diagnostics_df = pd.DataFrame(split_diagnostics)
display(split_diagnostics_df)

## Простая физически мотивированная модель

Физически мотивированная модель не является строгой тепловой
симуляцией. Она использует приближенный признак потерь, рассчитанный
из доступных измерений:

$$
P_{loss,proxy} = U I - M \omega.
$$

Далее линейная регрессия оценивает связь температуры с
температурой окружающей среды, прокси-потерями и охлаждением.

In [ ]:
def make_physics_features(data: pd.DataFrame) -> pd.DataFrame:
    omega = 2.0 * np.pi * data["speed_rpm"] / 60.0
    loss_proxy = data["voltage_v"] * data["current_a"] - data["torque_nm"] * omega
    return pd.DataFrame(
        {
            "ambient_temp_c": data["ambient_temp_c"],
            "loss_proxy_w": loss_proxy,
            "cooling_inverse": 1.0 / np.sqrt(data["cooling_air_speed_mps"].clip(lower=1.0)),
        },
        index=data.index,
    )

physics_model = LinearRegression()
physics_model.fit(make_physics_features(df.loc[train_idx]), y_train)
physics_pred = physics_model.predict(make_physics_features(df.loc[test_idx]))

In [ ]:
# TODO: впишите краткий текстовый ответ. объясните физический смысл P_loss_proxy = U I - M omega
physics_proxy_explanation = ""
if not physics_proxy_explanation.strip():
    raise ValueError('Заполните physics_proxy_explanation: объясните физический смысл P_loss_proxy = U I - M omega')
print(physics_proxy_explanation)

## Обучение моделей машинного обучения

In [ ]:
# TODO: задайте значение параметра. рекомендуемый диапазон max_depth случайного леса: 3..10
# Рекомендуемое значение для первого запуска: 8
forest_max_depth = None
if forest_max_depth is None:
    raise ValueError('Заполните forest_max_depth: рекомендуемый диапазон max_depth случайного леса: 3..10')

models = {
    "physics_proxy_linear": physics_model,
    "ridge": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]
    ),
    "random_forest": RandomForestRegressor(
        n_estimators=250,
        max_depth=forest_max_depth,
        min_samples_leaf=4,
        random_state=RANDOM_STATE,
    ),
    "gradient_boosting": GradientBoostingRegressor(
        n_estimators=180,
        learning_rate=0.045,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
}

predictions = {"physics_proxy_linear": physics_pred}
for name, model in models.items():
    if name == "physics_proxy_linear":
        continue
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)

metrics_rows = []
for name, pred in predictions.items():
    metrics_rows.append(
        {
            "model": name,
            "MAE_deg_C": mean_absolute_error(y_test, pred),
            "RMSE_deg_C": np.sqrt(mean_squared_error(y_test, pred)),
            "R2": r2_score(y_test, pred),
        }
    )
metrics_df = pd.DataFrame(metrics_rows).sort_values("RMSE_deg_C")
display(metrics_df)

In [ ]:
best_model_name = metrics_df.iloc[0]["model"]
best_pred = predictions[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, best_pred, alpha=0.70, s=20)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
axes[0].plot(lims, lims, color="black", linestyle="--")
axes[0].set_xlabel("Измеренная температура, deg_C")
axes[0].set_ylabel("Прогноз, deg_C")
axes[0].set_title(f"Лучшая модель: {best_model_name}")

residuals = y_test - best_pred
axes[1].scatter(df.loc[test_idx, "speed_rpm"], residuals, alpha=0.70, s=20)
axes[1].axhline(0.0, color="black", linestyle="--")
axes[1].set_xlabel("Частота вращения, rpm")
axes[1].set_ylabel("Остаток, deg_C")
axes[1].set_title("Остатки по частоте вращения")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: впишите краткий текстовый ответ. укажите лучшую модель по RMSE и объясните, почему одной метрики недостаточно
best_model_interpretation = ""
if not best_model_interpretation.strip():
    raise ValueError('Заполните best_model_interpretation: укажите лучшую модель по RMSE и объясните, почему одной метрики недостаточно')
print(best_model_interpretation)

## Анализ остатков

Остаток - разность между измеренной температурой и прогнозом.
Если остатки систематически зависят от высоты, тока или скорости
охлаждения, значит модель не полностью описывает соответствующий
физический фактор.

In [ ]:
residual_analysis_df = df.loc[test_idx, ["altitude_m", "current_a", "cooling_air_speed_mps"]].copy()
residual_analysis_df["residual_deg_C"] = y_test - best_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column, title in zip(
    axes,
    ["altitude_m", "current_a", "cooling_air_speed_mps"],
    ["Остаток по высоте", "Остаток по току", "Остаток по охлаждению"],
):
    ax.scatter(residual_analysis_df[column], residual_analysis_df["residual_deg_C"], s=20, alpha=0.70)
    ax.axhline(0.0, color="black", linestyle="--")
    ax.set_title(title)
    ax.set_xlabel(column)
    ax.set_ylabel("Остаток, deg_C")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: впишите краткий текстовый ответ. выберите один график остатков и кратко опишите, есть ли на нем систематическое смещение
residual_plot_interpretation = ""
if not residual_plot_interpretation.strip():
    raise ValueError('Заполните residual_plot_interpretation: выберите один график остатков и кратко опишите, есть ли на нем систематическое смещение')
print(residual_plot_interpretation)

In [ ]:
rf_model = models["random_forest"]
importance_df = pd.DataFrame(
    {
        "feature": thermal_features,
        "importance": rf_model.feature_importances_,
    }
).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(importance_df["feature"], importance_df["importance"], color="#59a14f")
ax.set_title("Важность признаков случайного леса")
ax.set_xlabel("Относительная важность")
plt.tight_layout()
plt.show()

display(importance_df.sort_values("importance", ascending=False))

## Перестановочная важность признаков

Перестановочная важность (permutation importance) показывает,
насколько ухудшается качество модели, если значения одного
признака случайно перемешать в тестовой выборке. Такой прием
удобен для объяснения: если после перемешивания признака ошибка
резко растет, значит модель сильно опиралась на этот признак.

In [ ]:
permutation_result = permutation_importance(
    rf_model,
    X_test,
    y_test,
    n_repeats=8,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
)
permutation_importance_df = pd.DataFrame(
    {
        "feature": thermal_features,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    }
).sort_values("importance_mean", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    permutation_importance_df["feature"],
    permutation_importance_df["importance_mean"],
    xerr=permutation_importance_df["importance_std"],
    color="#9c755f",
)
ax.set_title("Перестановочная важность признаков")
ax.set_xlabel("Рост RMSE при перемешивании признака, deg_C")
plt.tight_layout()
plt.show()
display(permutation_importance_df.sort_values("importance_mean", ascending=False))

## Демонстрация утечки данных

> **Внимание. АНТИПРИМЕР - НЕ ИСПОЛЬЗОВАТЬ КАК РАБОЧУЮ МОДЕЛЬ.**
> В следующей ячейке в признаки намеренно добавляется
> `steady_state_temp_c`. Этот столбец рассчитан из скрытой
> физической модели генератора и поэтому является диагностической
> подсказкой, а не обычным измеряемым признаком.

In [ ]:
leakage_df = df.merge(
    diagnostics_df[["sample_id", "steady_state_temp_c", "temperature_margin_c"]],
    on="sample_id",
    validate="one_to_one",
)
leakage_features = thermal_features + ["steady_state_temp_c"]

X_train_leak = leakage_df.loc[train_idx, leakage_features]
X_test_leak = leakage_df.loc[test_idx, leakage_features]
leakage_model = RandomForestRegressor(
    n_estimators=250,
    max_depth=8,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
)
leakage_model.fit(X_train_leak, y_train)
leakage_pred = leakage_model.predict(X_test_leak)
print("R2 строгой лучшей модели:", round(float(metrics_df.iloc[0]["R2"]), 4))
print("R2 модели с утечкой:", round(r2_score(y_test, leakage_pred), 4))

In [ ]:
# TODO: впишите краткий текстовый ответ. объясните, почему steady_state_temp_c нельзя использовать как базовый признак
thermal_leakage_explanation = ""
if not thermal_leakage_explanation.strip():
    raise ValueError('Заполните thermal_leakage_explanation: объясните, почему steady_state_temp_c нельзя использовать как базовый признак')
print(thermal_leakage_explanation)

## Самостоятельный эксперимент

In [ ]:
# TODO: задайте значение параметра. рекомендуемый диапазон max_depth: 3..10
# Рекомендуемое значение для первого запуска: 6
experiment_max_depth = None
if experiment_max_depth is None:
    raise ValueError('Заполните experiment_max_depth: рекомендуемый диапазон max_depth: 3..10')

experiment_model = RandomForestRegressor(
    n_estimators=160,
    max_depth=experiment_max_depth,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
)
experiment_model.fit(X_train, y_train)
experiment_pred = experiment_model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, experiment_pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, experiment_pred)), 3))
print("R2:", round(r2_score(y_test, experiment_pred), 4))

## Реестр найденных наборов данных и развернутые задания

В этом разделе используется единый реестр открытых источников для
занятий 4-6. Реестр не загружает крупные архивы автоматически.
Его назначение - показать, как переносить базовую учебную
постановку на реальные или открытые исследовательские источники.

Для занятия 4 студент должен выбрать один источник
из таблицы ниже и описать, как на нем можно воспроизвести логику
базового блокнота: определить признаки, целевую переменную или
скрытую разметку, способ разбиения выборки, риск утечки данных и
ожидаемые визуализации.

In [ ]:
dataset_catalog_04_06 = pd.read_csv(CATALOG_04_06_FILE)
dataset_assignments_04_06 = pd.read_csv(ASSIGNMENTS_04_06_FILE)

pd.set_option("display.max_colwidth", 120)
display(
    dataset_catalog_04_06[
        [
            "dataset_id",
            "name",
            "object",
            "lessons",
            "access",
            "risk_level",
            "implementation_status",
            "checked_at",
        ]
    ]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

risk_counts = dataset_catalog_04_06["risk_level"].value_counts()
axes[0].bar(risk_counts.index, risk_counts.values, color="#4c78a8")
axes[0].set_title("Уровень методического риска источников")
axes[0].set_ylabel("Число источников")

lesson_counts = (
    dataset_assignments_04_06["lesson"]
    .astype(str)
    .value_counts()
    .sort_index()
)
axes[1].bar(lesson_counts.index, lesson_counts.values, color="#f58518")
axes[1].set_title("Число развернутых заданий по занятиям 4-6")
axes[1].set_xlabel("Номер занятия")
axes[1].set_ylabel("Число заданий")

plt.tight_layout()
plt.show()

In [ ]:
lesson_assignments = dataset_assignments_04_06[
    dataset_assignments_04_06["lesson"].astype(str) == "4"
].copy()

display(
    lesson_assignments[
        [
            "assignment_id",
            "assignment_title",
            "implementation_status",
            "dataset_structure",
            "theory_block",
            "practice_block",
            "recommended_visualizations",
            "expected_artifacts",
            "control_questions",
            "risk_note",
        ]
    ]
)

## Индивидуальное расширенное задание

Выберите один источник из таблицы выше и заполните в отчете
отдельный подраздел:

1. объект исследования и единица наблюдения;
2. какие столбцы являются измеряемыми признаками;
3. какая величина является целевой переменной или скрытой
   диагностической разметкой;
4. какие столбцы нельзя использовать как признаки из-за риска
   утечки данных;
5. какой способ разбиения выборки является корректным;
6. какие 2-3 графика нужно построить в первую очередь.

Если полный архив не скачивался, это нужно явно указать. В таком
случае результатом считается методически корректная постановка
расширенного задания, а не численное обучение модели.

## Мини-задание по открытому источнику

NASA C-MAPSS Aircraft Engine Simulator Data - открытый набор
данных по моделированию деградации авиационного двигателя. Для
самостоятельного расширения не требуется скачивать архив прямо
сейчас. В отчете укажите:

1. какие столбцы могли бы быть признаками;
2. какую величину можно считать целевой переменной;
3. почему для такого источника опасно случайно перемешивать
   соседние временные точки;
4. какую метрику качества следует использовать для прогноза
   температуры или остаточного ресурса.

## Задание для отчета

1. Объясните, почему случайное разбиение нужно сравнивать с
   групповым разбиением по профилям.
2. Сравните физически мотивированную модель, Ridge-регрессию,
   случайный лес и градиентный бустинг.
3. Укажите две наиболее важные величины для прогноза температуры.
4. Объясните, почему важность признака в случайном лесе не равна
   строгой физической причинности.
5. Опишите антипример утечки данных и не используйте его метрики
   как основной результат.
6. Выполните мини-задание по NASA C-MAPSS на уровне постановки
   задачи без загрузки архива.

Открытые источники для расширения: NASA C-MAPSS Aircraft Engine
Simulator Data и открытые наборы температурного моделирования
электромеханических систем. Их следует применять только после
отдельной проверки структуры временных рядов и правил разбиения.